In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import entropy as scipy_entropy
from tqdm import tqdm
from transformers import AutoProcessor

import os
from vla.constants import PROJECT_ROOT
from vla.models.smolvla import SmolVLAPolicy


In [ ]:
task = "giraffe"
image = Image.open(os.path.join(PROJECT_ROOT, "notebooks", "giraffe.jpg")).convert("RGB")
image

In [ ]:
checkpoint = "HuggingFaceVLA/smolvla_libero"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from checkpoint: {checkpoint} on device: {device}")
policy = SmolVLAPolicy(checkpoint=checkpoint, device=device, action_dim=7)

In [ ]:
# %% Setup: processor, hooks, and model references (done ONCE)
hf_vlm = policy.model.vlm_with_expert.vlm
llm = hf_vlm.model.text_model
total_layers = len(llm.layers)
layer_indices = list(range(total_layers // 2))  # SmolVLA discards final 50%
vision_dtype = next(hf_vlm.model.vision_model.parameters()).dtype

# Force eager attention output
llm.config._attn_implementation = "eager"
llm.config.output_attentions = True

# Load processor ONCE
try:
    processor = AutoProcessor.from_pretrained("HuggingFaceVLA/smolvla_libero")
except Exception:
    processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-500M-Instruct")

# Pre-compute the text prompt template (same for every frame)
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": task}]}]
prompt_template = processor.apply_chat_template(messages, add_generation_prompt=True)

print(f"Model ready — {total_layers} total layers, using first {len(layer_indices)}")
print(f"Processor loaded, prompt template cached.")


In [ ]:
def get_multi_layer_attention(model, pil_image, task_description, device="cuda"):
    """
    Hooks into multiple percentiles of the LLM to track attention evolution.
    """
    attention_maps = {}

    # Language Model
    hf_vlm = policy.model.vlm_with_expert.vlm
    llm = hf_vlm.model.text_model
    total_layers = len(llm.layers)
    print(f"Isolated the Language Model successfully. Total layers: {total_layers}")

    # layer choice
    layer_indices = "all"
    if layer_indices == "all":
        # SmolVLA takes inspiration from nvidia's work showing that only half the layers are needed for good attention 
        # maps so they discared final 50% of layes, also mentioned in paper
        layer_indices = list(range(total_layers//2)) 
    print(f"Targeting layer indices: {layer_indices}")

    # Create a unique hook for each layer
    def get_forward_hook(layer_idx):
        def hook(module, input, output):
            print(f"Hook triggered for layer {layer_idx}!")
            # output[1] contains the attention weights
            attention_maps[layer_idx] = output[1].detach().cpu()

        return hook

    # Register hooks on all target layers
    hook_handles = []
    for idx in layer_indices:
        target_layer = llm.layers[idx].self_attn
        handle = target_layer.register_forward_hook(get_forward_hook(idx))
        hook_handles.append(handle)

    # Force the LLM to output attention
    llm.config._attn_implementation = "eager"
    llm.config.output_attentions = True

    # Process the Image and Text together
    print("Processing image and text prompt...")
    # FIX: Use the processor from the CHECKPOINT, not the base model.
    try:
        processor = AutoProcessor.from_pretrained("HuggingFaceVLA/smolvla_libero")
    except Exception:
        print("Could not load processor from checkpoint, falling back to base.")
        processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-500M-Instruct")

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": task_description}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=pil_image, return_tensors="pt").to(device)

    # Check input resolution
    print(f"Input image shape: {inputs['pixel_values'].shape}")

    # Ensure image dtype matches vision model
    vision_dtype = next(hf_vlm.model.vision_model.parameters()).dtype
    inputs["pixel_values"] = inputs["pixel_values"].to(vision_dtype)

    # Run forward pass on the FULL VLM
    print("Running forward pass through the VLM...")
    with torch.no_grad():
        _ = hf_vlm(**inputs)

    # Clean up hooks
    for handle in hook_handles:
        handle.remove()

    return attention_maps, inputs, layer_indices, processor

In [ ]:
def visualize_multi_layer_heatmap(attention_maps_dict, inputs, pil_image, layer_indices, processor, target_word,
                                   n_cols=6):
    """
    Uses all 16 local crops (each 8x8 = 64 tokens) to build a spatially accurate
    32x32 attention grid over the full image, rather than the single global crop.
    Plots in a grid of n_cols columns, with as many rows as needed.
    The original image occupies the first cell.
    A grid overlay shows the 16 crop boundaries to help spot cropping artifacts.
    """
    # Find the target token index
    input_ids = inputs["input_ids"][0].tolist()
    tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)

    target_idx = -1
    for i, t in enumerate(tokens):
        if target_word.lower() in t.lower():
            target_idx = i
            print(f"Tracking attention for word '{t}' (token index {i})")
            break

    if target_idx == -1:
        print(f"Warning: Could not find '{target_word}'. Defaulting to the last token.")
        target_idx = len(input_ids) - 1

    # Find image token indices
    most_common_token = Counter(input_ids).most_common(1)[0][0]
    image_token_indices = [i for i, token in enumerate(input_ids) if token == most_common_token]
    num_crops = inputs["pixel_values"].shape[1]
    tokens_per_crop = len(image_token_indices) // num_crops  # always 64
    grid_size = int(np.sqrt(tokens_per_crop))  # 8

    # Build list of token indices per local crop (exclude the last global crop)
    num_local_crops = num_crops - 1  # e.g. 16
    crop_token_indices = [
        image_token_indices[c * tokens_per_crop : (c + 1) * tokens_per_crop]
        for c in range(num_local_crops)
    ]

    # Infer the spatial arrangement of local crops (e.g. 4x4 for 16 crops)
    n_crop_cols = int(np.sqrt(num_local_crops))
    n_crop_rows = int(np.ceil(num_local_crops / n_crop_cols))
    full_grid_h = n_crop_rows * grid_size  # 32
    full_grid_w = n_crop_cols * grid_size  # 32
    print(f"Assembling {num_local_crops} local crops into a {n_crop_rows}x{n_crop_cols} grid "
          f"→ {full_grid_h}x{full_grid_w} attention grid")

    # Crop boundary positions in image pixel space
    crop_w = pil_image.width / n_crop_cols
    crop_h = pil_image.height / n_crop_rows

    def draw_crop_grid(ax):
        """Draw grid lines at crop boundaries."""
        for col in range(1, n_crop_cols):
            ax.axvline(x=col * crop_w, color='white', linewidth=0.8, linestyle='--', alpha=0.7)
        for row in range(1, n_crop_rows):
            ax.axhline(y=row * crop_h, color='white', linewidth=0.8, linestyle='--', alpha=0.7)

    # Grid layout: original image + one cell per layer
    num_plots = len(layer_indices) + 1  # +1 for original image
    n_rows = int(np.ceil(num_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows), squeeze=False)
    axes_flat = axes.flatten()

    # Plot original image in first cell (with grid only, no heatmap)
    axes_flat[0].imshow(pil_image)
    draw_crop_grid(axes_flat[0])
    axes_flat[0].set_title("Original", fontsize=8)
    axes_flat[0].axis("off")

    # Process and plot each layer
    for plot_idx, layer_idx in enumerate(layer_indices):
        # Average across attention heads
        attn = attention_maps_dict[layer_idx].squeeze(0).mean(dim=0)
        word_attn = attn[target_idx, :]

        # Stitch the 16 local crops into the full spatial grid
        full_grid = np.zeros((full_grid_h, full_grid_w), dtype=np.float32)
        for c, crop_indices in enumerate(crop_token_indices):
            crop_row = c // n_crop_cols
            crop_col = c % n_crop_cols
            crop_attn = word_attn[crop_indices]
            if crop_attn.dtype == torch.bfloat16:
                crop_attn = crop_attn.to(torch.float32)
            patch = crop_attn.numpy().reshape(grid_size, grid_size)
            full_grid[
                crop_row * grid_size : (crop_row + 1) * grid_size,
                crop_col * grid_size : (crop_col + 1) * grid_size,
            ] = patch

        # Clip to reduce attention sink dominance, then normalise
        vmax = np.percentile(full_grid, 80)
        vmin = np.percentile(full_grid, 2)
        full_grid = np.clip(full_grid, vmin, vmax)
        full_grid = (full_grid - full_grid.min()) / (full_grid.max() - full_grid.min() + 1e-8)

        ax = axes_flat[plot_idx + 1]
        ax.imshow(pil_image)
        ax.imshow(full_grid, cmap='jet', alpha=0.5,
                  extent=[0, pil_image.width, pil_image.height, 0])
        draw_crop_grid(ax)
        ax.set_title(f"Layer {layer_idx}", fontsize=8)
        ax.axis("off")

    # Hide any unused cells
    for i in range(num_plots, len(axes_flat)):
        axes_flat[i].axis("off")

    plt.suptitle(f"Attention heatmaps — '{target_word}' (16 local crops)", fontsize=10, y=1.01)
    plt.tight_layout()
    plt.show()


In [ ]:
attention_maps_dict, model_inputs, layer_indices, processor = get_multi_layer_attention(
    policy, image, task, device
)

In [ ]:
print("Generating multi-layer heatmap overlay...")
visualize_multi_layer_heatmap(attention_maps_dict, model_inputs, image, layer_indices, processor, task)

In [ ]:
def attention_rollout(attentions):
    # Initialize rollout with identity matrix
    rollout = torch.eye(attentions[0].size(-1), dtype=attentions[0].dtype, device=attentions[0].device)

    # Multiply attention maps layer by layer
    for attention in attentions:
        attention_heads_fused = attention.mean(dim=1) # Average attention across heads
        attention_heads_fused += torch.eye(attention_heads_fused.size(-1), dtype=attention_heads_fused.dtype, device=attention_heads_fused.device) # A + I
        attention_heads_fused /= attention_heads_fused.sum(dim=-1, keepdim=True) # Normalizing A
        rollout = torch.matmul(rollout, attention_heads_fused) # Multiplication

    return rollout

In [ ]:
# Use attention tensors in layer order (not dict keys directly)
attentions = [attention_maps_dict[i] for i in sorted(attention_maps_dict)]

rollout = attention_rollout(attentions)
rollout = rollout.squeeze(0)  # optional: remove batch dim if present
print("Rollout shape:", rollout.shape)

In [ ]:
from collections import Counter

input_ids = model_inputs["input_ids"][0].tolist()
tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)
target_word = task
target_idx = -1
for i, t in enumerate(tokens):
    if target_word.lower() in t.lower():
        target_idx = i
        break

if target_idx == -1:
    target_idx = len(input_ids) - 1

most_common_token = Counter(input_ids).most_common(1)[0][0]
image_token_indices = [i for i, token in enumerate(input_ids) if token == most_common_token]

num_crops = model_inputs["pixel_values"].shape[1]
tokens_per_crop = len(image_token_indices) // num_crops
grid_size = int(np.sqrt(tokens_per_crop))

num_local_crops = num_crops - 1
crop_token_indices = [
    image_token_indices[c * tokens_per_crop : (c + 1) * tokens_per_crop]
    for c in range(num_local_crops)
]

n_crop_cols = int(np.sqrt(num_local_crops))
n_crop_rows = int(np.ceil(num_local_crops / n_crop_cols))
full_grid_h = n_crop_rows * grid_size
full_grid_w = n_crop_cols * grid_size

target_attention = rollout[target_idx, :]

full_grid = np.zeros((full_grid_h, full_grid_w), dtype=np.float32)
for c, crop_indices in enumerate(crop_token_indices):
    crop_row = c // n_crop_cols
    crop_col = c % n_crop_cols
    crop_attn = target_attention[crop_indices]
    if crop_attn.dtype == torch.bfloat16:
        crop_attn = crop_attn.to(torch.float32)
    patch = crop_attn.cpu().numpy().reshape(grid_size, grid_size)
    full_grid[
        crop_row * grid_size : (crop_row + 1) * grid_size,
        crop_col * grid_size : (crop_col + 1) * grid_size,
    ] = patch

rollout_attention = full_grid

In [ ]:
from PIL import ImageFilter

# Normalize the attention map for better visualization
# Clip to reduce attention sink dominance, matching the initial visualization
vmax = np.percentile(rollout_attention, 80)
vmin = np.percentile(rollout_attention, 5)
rollout_attention = np.clip(rollout_attention, vmin, vmax)
rollout_attention = (rollout_attention - rollout_attention.min()) / (rollout_attention.max() - rollout_attention.min() + 1e-8)

img_w, img_h = image.size
# Resize and blur the attention map
rollout_attention_img = Image.fromarray((rollout_attention * 255).astype(np.uint8)).resize((img_w, img_h), resample=Image.BICUBIC)
rollout_attention_img = rollout_attention_img.filter(ImageFilter.GaussianBlur(radius=2))

In [ ]:
# Convert the attention map to a heatmap
import matplotlib.cm as cm

# Create a colormap
colormap = cm.get_cmap('jet')
colored_attention = colormap(np.array(rollout_attention_img) / 255.0)

# Convert to an RGBA image
attention_heatmap = Image.fromarray((colored_attention * 255).astype(np.uint8), mode="RGBA")
attention_heatmap.putalpha(150) # Adjust alpha for blending

# Overlay on the original image
result_img = Image.alpha_composite(image.convert("RGBA"), attention_heatmap)
result_img

---